In [ ]:
pip install gensim


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 74.7 MB/s eta 0:00:00


In [ ]:
import pandas as pd
import numpy as np
import random
from sklearn.model_selection import train_test_split
from sklearn.metrics.pairwise import cosine_similarity
import gensim.downloader as api

# ------------------ Reproducibility ------------------
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# ------------------ Load Dataset ------------------
PATH = "Dataset_GitHub_SO_v04.xlsx"
df = pd.read_excel(PATH)
required = {'commit_id','so_id','p_text','s_text','label'}
assert required.issubset(df.columns)
df['label'] = df['label'].astype(int)

# Commit-level 80/20 split
commit_ids = df.commit_id.unique()
train_cids, test_cids = train_test_split(commit_ids, test_size=0.2, random_state=SEED)

df_train = df[df.commit_id.isin(train_cids)].copy()
df_test  = df[df.commit_id.isin(test_cids)].copy()

# Unique SO posts (target artifacts)
df_s = df[['so_id','s_text']].drop_duplicates().reset_index(drop=True)

# Unique commits/issues (source artifacts)
df_g = df[['commit_id','p_text']].drop_duplicates().reset_index(drop=True)

In [ ]:
# ------------------ Load Word2Vec ------------------
wv = api.load("word2vec-google-news-300")



# WE

In [ ]:
# WE Baseline Implementation
class WEBaseline:
    def __init__(self, df_s, wv_model):
        self.df_s = df_s.copy()
        self.wv = wv_model
        self.embedding_dim = self.wv.vector_size
        self.s_vecs = np.array([self.vectorize(text) for text in self.df_s['s_text']])

    def vectorize(self, text):
        words = text.split()
        vecs = [self.wv[w] for w in words if w in self.wv]
        return np.mean(vecs, axis=0) if vecs else np.zeros(self.embedding_dim)

    def rank_solutions(self, p_text, top_k=5):
        p_vec = self.vectorize(p_text).reshape(1, -1)
        sim_scores = cosine_similarity(p_vec, self.s_vecs).flatten()
        top_idx = np.argsort(sim_scores)[::-1][:top_k]
        return self.df_s.iloc[top_idx].assign(
            Score = sim_scores[top_idx],
            Rank = np.arange(1, top_k+1)
        )

# Initialize WE baseline
we_baseline = WEBaseline(df_s, wv)

# ------------------ Evaluation Function ------------------
def evaluate_we_baseline(baseline, df, top_k_list=[1,3,5,10,15]):
    results = {f"HR@{k}": 0 for k in top_k_list}
    mrr = 0
    for cid, group in df.groupby("commit_id"):
        p_text = group['p_text'].iloc[0]
        ranked_df = baseline.rank_solutions(p_text, top_k=max(top_k_list))
        y_true = np.array([1 if so in group[group.label==1]['so_id'].values else 0 for so in ranked_df['so_id']])
        for k in top_k_list:
            results[f"HR@{k}"] += int(y_true[:k].sum() > 0)
        hits = np.where(y_true==1)[0]
        if len(hits) > 0:
            mrr += 1.0 / (hits[0]+1)
    n_commits = df['commit_id'].nunique()
    results = {k: v/n_commits for k,v in results.items()}
    mrr /= n_commits
    return results, mrr

    # ------------------ Evaluate on Test Set ------------------
we_metrics, we_mrr = evaluate_we_baseline(we_baseline, df_test)
print("WE Baseline Results:")
print(we_metrics)
print("MRR:", we_mrr)


In [ ]:
# ------------------ Evaluate on Test Set ------------------
we_metrics, we_mrr = evaluate_we_baseline(we_baseline, df_test)
print("WE Baseline Results:")
print(we_metrics)
print("MRR:", we_mrr)

WE Baseline Results:
{'HR@1': 0.05583756345177665, 'HR@3': 0.10152284263959391, 'HR@5': 0.12690355329949238, 'HR@10': 0.14720812182741116, 'HR@15': 0.17258883248730963}
MRR: 0.08536707229600629


# LSI

In [ ]:
## LSI Baseline Implementation
from sklearn.decomposition import TruncatedSVD
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
import pandas as pd

# ------------------ LSI Baseline Class ------------------
class LSIBaseline:
    def __init__(self, df_s, n_components=300):
        """
        df_s: DataFrame of candidate SO posts ['so_id','s_text']
        n_components: number of latent dimensions for LSI
        """
        self.df_s = df_s.copy()
        self.vectorizer = TfidfVectorizer(stop_words='english', max_features=5000)
        tfidf_matrix = self.vectorizer.fit_transform(self.df_s['s_text'])

        # Apply Truncated SVD to get latent semantic space
        self.svd = TruncatedSVD(n_components=n_components, random_state=42)
        self.lsi_matrix = self.svd.fit_transform(tfidf_matrix)

    def rank_solutions(self, p_text, top_k=5):
        """
        Rank candidate solutions for a given problem text
        """
        p_vec = self.vectorizer.transform([p_text])
        p_lsi = self.svd.transform(p_vec)

        sim_scores = cosine_similarity(p_lsi, self.lsi_matrix).flatten()
        top_idx = np.argsort(sim_scores)[::-1][:top_k]

        return self.df_s.iloc[top_idx].assign(
            Score=sim_scores[top_idx],
            Rank=np.arange(1, top_k+1)
        )

# ------------------ Usage ------------------
lsi_baseline = LSIBaseline(df_s, n_components=300)

# Evaluate using the same evaluation function used for WE
lsi_metrics, lsi_mrr = evaluate_we_baseline(lsi_baseline, df_test)
print("LSI Baseline Results:")
print(lsi_metrics)
print("MRR:", lsi_mrr)


LSI Baseline Results:
{'HR@1': 0.25888324873096447, 'HR@3': 0.40609137055837563, 'HR@5': 0.467005076142132, 'HR@10': 0.5279187817258884, 'HR@15': 0.6142131979695431}
MRR: 0.3550168528594926


# CrossLink

In [ ]:
#CrossLink baseline implimentation
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# ------------------ CrossLink Baseline ------------------
class CrossLinkBaseline:
    """
    Adapted CrossLink baseline for linking commits/issues (problems)
    to SO posts (solutions) using textual similarity (TF-IDF + cosine similarity).
    """
    def __init__(self, df_s):
        """
        df_s: DataFrame of candidate SO posts ['so_id', 's_text']
        """
        self.df_s = df_s.copy()
        self.vectorizer = TfidfVectorizer(stop_words='english', max_features=5000)
        self.s_tfidf = self.vectorizer.fit_transform(self.df_s['s_text'].values)

    def rank_solutions(self, p_text, top_k=5):
        """
        Rank candidate solutions for a given problem text.
        """
        p_vec = self.vectorizer.transform([p_text])
        sim_scores = cosine_similarity(p_vec, self.s_tfidf).flatten()
        top_idx = np.argsort(sim_scores)[::-1][:top_k]
        return self.df_s.iloc[top_idx].assign(
            Score = sim_scores[top_idx],
            Rank = np.arange(1, top_k+1)
        )

# ------------------ Evaluation Function (reuse previous one) ------------------
def evaluate_crosslink_baseline(baseline, df, top_k_list=[1,3,5,10,15]):
    """
    Evaluate CrossLink baseline using Hit@K and MRR.
    """
    results = {f"HR@{k}": 0 for k in top_k_list}
    mrr = 0

    for cid, group in df.groupby("commit_id"):
        p_text = group['p_text'].iloc[0]
        ranked_df = baseline.rank_solutions(p_text, top_k=max(top_k_list))

        y_true = [1 if so in group[group.label==1]['so_id'].values else 0 for so in ranked_df['so_id']]
        y_true = np.array(y_true)

        for k in top_k_list:
            results[f"HR@{k}"] += int(y_true[:k].sum() > 0)

        hits = np.where(y_true == 1)[0]
        if len(hits) > 0:
            mrr += 1.0 / (hits[0] + 1)

    n_commits = df['commit_id'].nunique()
    results = {k: v/n_commits for k,v in results.items()}
    mrr /= n_commits

    return results, mrr

# ------------------ Usage Example ------------------
crosslink_baseline = CrossLinkBaseline(df_s)
crosslink_metrics, crosslink_mrr = evaluate_crosslink_baseline(crosslink_baseline, df_test)

print("CrossLink Baseline Results:")
print(crosslink_metrics)
print("MRR:", crosslink_mrr)


CrossLink Baseline Results:
{'HR@1': 0.3248730964467005, 'HR@3': 0.4873096446700508, 'HR@5': 0.5431472081218274, 'HR@10': 0.6395939086294417, 'HR@15': 0.700507614213198}
MRR: 0.42812594906503537


# E-mailCodeLinker

In [ ]:
# E-mailCodeLinker Baseline Implementation
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
import pandas as pd

class EmailCodeLinkerBaseline:
    def __init__(self, df_s, max_features=5000):
        """
        E-mailCodeLinker adapted to link GitHub commits/issues to SO posts.

        df_s: DataFrame of candidate SO posts ['so_id', 's_text']
        max_features: maximum number of TF-IDF features
        """
        self.df_s = df_s.copy()
        self.vectorizer = TfidfVectorizer(stop_words='english', max_features=max_features)
        # Fit TF-IDF on all SO posts
        self.s_tfidf = self.vectorizer.fit_transform(self.df_s['s_text'].values)

    def rank_solutions(self, p_text, top_k=5):
        """
        Rank candidate solutions for a given architectural problem.
        p_text: string of the problem
        top_k: number of top results to return

        Returns: DataFrame with columns ['so_id','s_text','Score','Rank']
        """
        # Transform the problem text into TF-IDF space
        p_vec = self.vectorizer.transform([p_text])
        # Compute cosine similarity with all SO posts
        sim_scores = cosine_similarity(p_vec, self.s_tfidf).flatten()
        # Get top-k indices
        top_idx = np.argsort(sim_scores)[::-1][:top_k]

        results = pd.DataFrame({
            'so_id': self.df_s.iloc[top_idx]['so_id'].values,
            's_text': self.df_s.iloc[top_idx]['s_text'].values,
            'Score': sim_scores[top_idx],
            'Rank': np.arange(1, top_k+1)
        })
        return results

# ------------------ Usage Example ------------------
emailcode_baseline = EmailCodeLinkerBaseline(df_s)

# Pick a commit/problem example
commit_example = df_g['p_text'].iloc[0]
ranked_solutions = emailcode_baseline.rank_solutions(commit_example, top_k=5)
print(ranked_solutions)

# ------------------ Evaluation ------------------
def evaluate_emailcode_baseline(baseline, df, top_k_list=[1,3,5,10,15]):
    """
    Evaluate the E-mailCodeLinker baseline using Hit@K and MRR.

    baseline: object with method rank_solutions(p_text, top_k)
    df: DataFrame containing ['commit_id','p_text','so_id','label']
    top_k_list: list of K values for Hit@K
    """
    results = {f"HR@{k}": 0 for k in top_k_list}
    mrr = 0

    for cid, group in df.groupby("commit_id"):
        p_text = group['p_text'].iloc[0]  # Architectural problem text
        ranked_df = baseline.rank_solutions(p_text, top_k=max(top_k_list))

        # Map ranked SO posts to ground truth
        y_true = [1 if so in group[group.label==1]['so_id'].values else 0 for so in ranked_df['so_id']]
        y_true = np.array(y_true)

        # Compute Hit@K
        for k in top_k_list:
            results[f"HR@{k}"] += int(y_true[:k].sum() > 0)

        # Compute MRR
        hits = np.where(y_true == 1)[0]
        if len(hits) > 0:
            mrr += 1.0 / (hits[0] + 1)

    n_commits = df['commit_id'].nunique()
    results = {k: v/n_commits for k,v in results.items()}
    mrr /= n_commits

    return results, mrr

# Evaluate on test set
emailcode_metrics, emailcode_mrr = evaluate_emailcode_baseline(emailcode_baseline, df_test)
print("E-mailCodeLinker Baseline Results:")
print(emailcode_metrics)
print("MRR:", emailcode_mrr)


    so_id                                             s_text     Score  Rank
0  SO_969  <p>When you are using <a href="http://www.djan...  0.365088     1
1  SO_512  How do you deal with validation on complex agg...  0.338850     2
2  SO_318  I have started using aspects for parameter val...  0.275784     3
3  SO_617  As of Django 3.0 you can use default\ncreatesu...  0.229715     4
4   SO_63  You may create a custom annotation to validate...  0.223425     5
E-mailCodeLinker Baseline Results:
{'HR@1': 0.3248730964467005, 'HR@3': 0.4873096446700508, 'HR@5': 0.5431472081218274, 'HR@10': 0.6395939086294417, 'HR@15': 0.700507614213198}
MRR: 0.42812594906503537


# ProgrammingService

In [ ]:
#ProgrammingService baseline implimentation
!pip install sentence-transformers

from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import pandas as pd
import numpy as np

# ------------------ Programming Service Baseline ------------------
class ProgrammingServiceBaseline:
    def __init__(self, df_s, model_name='all-MiniLM-L6-v2'):
        """
        df_s: DataFrame of candidate SO posts ['so_id', 's_text']
        model_name: sentence-transformer model for embeddings
        """
        self.df_s = df_s.copy()
        self.model = SentenceTransformer(model_name)

        # Precompute embeddings for all SO posts
        self.s_vecs = self.model.encode(self.df_s['s_text'].tolist(), convert_to_numpy=True)

    def rank_solutions(self, p_text, top_k=5):
        """
        Rank candidate SO posts for a given problem text
        """
        p_vec = self.model.encode([p_text], convert_to_numpy=True)
        sim_scores = cosine_similarity(p_vec, self.s_vecs).flatten()
        top_idx = np.argsort(sim_scores)[::-1][:top_k]

        return self.df_s.iloc[top_idx].assign(
            Score=sim_scores[top_idx],
            Rank=np.arange(1, top_k+1)
        )

# ------------------ Evaluation Function (same as WE) ------------------
def evaluate_baseline(baseline, df, top_k_list=[1,3,5,10,15]):
    results = {f"HR@{k}": 0 for k in top_k_list}
    mrr = 0

    for cid, group in df.groupby("commit_id"):
        p_text = group['p_text'].iloc[0]
        ranked_df = baseline.rank_solutions(p_text, top_k=max(top_k_list))

        # Map ranked SO posts to ground truth
        y_true = [1 if so in group[group.label==1]['so_id'].values else 0 for so in ranked_df['so_id']]
        y_true = np.array(y_true)

        # Hit@K
        for k in top_k_list:
            results[f"HR@{k}"] += int(y_true[:k].sum() > 0)

        # MRR
        hits = np.where(y_true==1)[0]
        if len(hits) > 0:
            mrr += 1.0 / (hits[0] + 1)

    n_commits = df['commit_id'].nunique()
    results = {k: v/n_commits for k,v in results.items()}
    mrr /= n_commits
    return results, mrr

In [ ]:
# ------------------ Usage Example ------------------
programming_service = ProgrammingServiceBaseline(df_s)
ps_metrics, ps_mrr = evaluate_baseline(programming_service, df_test)

print("Programming Service Baseline Results:")
print(ps_metrics)
print("MRR:", ps_mrr)

Programming Service Baseline Results:
{'HR@1': 0.5634517766497462, 'HR@3': 0.766497461928934, 'HR@5': 0.8223350253807107, 'HR@10': 0.8934010152284264, 'HR@15': 0.9187817258883249}
MRR: 0.6751350172669969


# EMRCM Baseline

In [ ]:
# EMRCM Baseline Implementation
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
import pandas as pd

class EMRCMBaseline:
    def __init__(self, df_s):
        """
        df_s: DataFrame of candidate SO posts ['so_id','s_text']
        """
        self.df_s = df_s.copy()
        self.vectorizer = TfidfVectorizer(stop_words='english', max_features=5000)
        # Fit TF-IDF on all SO posts
        self.s_vecs = self.vectorizer.fit_transform(self.df_s['s_text'].values)

    def rank_solutions(self, p_text, top_k=5):
        """
        Rank candidate solutions for a given problem text.
        p_text: string of the architectural problem
        top_k: number of top results to return
        Returns: DataFrame with columns ['so_id','s_text','Score','Rank']
        """
        # Transform problem text into TF-IDF space
        p_vec = self.vectorizer.transform([p_text])
        # Compute cosine similarity with all SO posts
        sim_scores = cosine_similarity(p_vec, self.s_vecs).flatten()
        # Get top-k indices
        top_idx = np.argsort(sim_scores)[::-1][:top_k]

        results = pd.DataFrame({
            'so_id': self.df_s.iloc[top_idx]['so_id'].values,
            's_text': self.df_s.iloc[top_idx]['s_text'].values,
            'Score': sim_scores[top_idx],
            'Rank': np.arange(1, top_k+1)
        })
        return results

# ==========================================================
# Initialize EMRCM baseline
# ==========================================================
emrcm_baseline = EMRCMBaseline(df_s)

# ==========================================================
# Reuse the same evaluation function as other baselines
# ==========================================================
emrcm_metrics, emrcm_mrr = evaluate_we_baseline(emrcm_baseline, df_test)
print("EMRCM Baseline Results:")
print(emrcm_metrics)
print("MRR:", emrcm_mrr)


EMRCM Baseline Results:
{'HR@1': 0.3248730964467005, 'HR@3': 0.4873096446700508, 'HR@5': 0.5431472081218274, 'HR@10': 0.6395939086294417, 'HR@15': 0.700507614213198}
MRR: 0.42812594906503537


# Lucene

In [ ]:
"""Baseline_Lucene_10-K_Fold.ipynb
"""

!apt-get update -qq
!apt-get install -y openjdk-21-jdk

import os

# JAVA_HOME
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-21-openjdk-amd64"

# Path to libjvm.so for Pyjnius
os.environ["JVM_PATH"] = "/usr/lib/jvm/java-21-openjdk-amd64/lib/server/libjvm.so"

# Add JAVA_HOME/bin to PATH
os.environ["PATH"] = os.environ["JAVA_HOME"] + "/bin:" + os.environ["PATH"]

# Check versions
!java -version
!echo $JAVA_HOME
!ls $JAVA_HOME/lib/server/libjvm.so

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following additional packages will be installed:
  at-spi2-core fonts-dejavu-core fonts-dejavu-extra gsettings-desktop-schemas
  libatk-bridge2.0-0 libatk-wrapper-java libatk-wrapper-java-jni libatk1.0-0
  libatk1.0-data libatspi2.0-0 libgtk-3-0 libgtk-3-bin libgtk-3-common
  librsvg2-common libxcomposite1 libxt-dev libxtst6 libxxf86dga1
  openjdk-21-jdk-headless openjdk-21-jre openjdk-21-jre-headless
  session-migration x11-utils
Suggested packages:
  gvfs libxt-doc openjdk-21-demo openjdk-21-source visualvm libnss-mdns
  fonts-ipafont-gothic fonts-ipafont-mincho fonts-wqy-microhei
  | fonts-wqy-zenhei fonts-indic mesa-utils
The following NEW packages will be installed:
  at-spi2-core fonts-dejavu-co

In [ ]:
!pip install -q pyserini

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 159.7/159.7 MB 7.2 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 633.7/633.7 kB 43.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.1/17.1 MB 98.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.0/7.0 MB 124.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 85.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 200.5/200.5 kB 22.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 96.4/96.4 kB 10.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 152.3/152.3 kB 16.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 82.0/82.0 kB 8.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 331.1/331.1 kB 33.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account

In [ ]:
import numpy as np
import pandas as pd
import random
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import KFold

# ------------------ Reproducibility ------------------
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# ==========================================================
# Load Dataset
# ==========================================================
PATH = "Dataset_GitHub_Solutions.xlsx"  # adjust path
df = pd.read_excel(PATH)

required_cols = {'commit_id','so_id','p_text','s_text','label'}
assert required_cols.issubset(df.columns), "Missing required columns"

df['label'] = df['label'].astype(int)
df['p_text'] = df['p_text'].fillna('').str.lower().str.strip()
df['s_text'] = df['s_text'].fillna('').str.lower().str.strip()
df['so_id'] = df['so_id'].astype(str)

commit_ids = df.commit_id.unique()
n_splits = 10
top_ks = [1, 3, 5, 10, 15]
top_m = max(top_ks)

# ==========================================================
# TF-IDF Baseline
# ==========================================================
class LuceneBaseline:
    def __init__(self, df_s_all):
        # All SO posts are indexed to avoid missing relevant ones
        self.df_s = df_s_all.reset_index(drop=True)
        self.vectorizer = TfidfVectorizer(stop_words='english', max_features=20000)
        self.tfidf_matrix = self.vectorizer.fit_transform(self.df_s['s_text'].values)

    def rank(self, p_text, top_k=5):
        p_vec = self.vectorizer.transform([p_text])
        sims = cosine_similarity(p_vec, self.tfidf_matrix).flatten()
        top_idx = np.argsort(sims)[::-1][:top_k]
        return self.df_s.iloc[top_idx][['so_id']].assign(
            score=sims[top_idx],
            rank=np.arange(1, len(top_idx)+1)
        )

# ==========================================================
# 10-Fold CV Evaluation
# ==========================================================
kf = KFold(n_splits=n_splits, shuffle=True, random_state=SEED)
fold_metrics = []

# Use all unique SO posts for TF-IDF index
df_s_all = df[['so_id','s_text']].drop_duplicates().reset_index(drop=True)

for fold_id, (train_idx, test_idx) in enumerate(kf.split(commit_ids)):
    print(f"[Lucene] Fold {fold_id+1}/{n_splits}")

    train_cids = commit_ids[train_idx]
    test_cids  = commit_ids[test_idx]

    df_train = df[df.commit_id.isin(train_cids)]
    df_test  = df[df.commit_id.isin(test_cids)]

    # Initialize TF-IDF baseline with all SO posts
    lucene = LuceneBaseline(df_s_all)

    hr = {f'HR@{k}': 0 for k in top_ks}
    mrr = 0.0

    # Evaluate per commit
    for cid, group in df_test.groupby('commit_id'):
        p_text = group['p_text'].iloc[0]
        relevant_so = set(group[group.label == 1]['so_id'].values)

        ranked_df = lucene.rank(p_text, top_k=top_m)
        ranked_so = ranked_df['so_id'].astype(str).values

        # HR@K
        for k in top_ks:
            topk_so = ranked_so[:k]
            hr[f'HR@{k}'] += int(any(so in relevant_so for so in topk_so))

        # MRR
        for rank_idx, so in enumerate(ranked_so, start=1):
            if so in relevant_so:
                mrr += 1.0 / rank_idx
                break

    n_commits = df_test['commit_id'].nunique()
    # Normalize HR@K and MRR
    fold_result = {k: v/n_commits for k,v in hr.items()}
    fold_result['MRR'] = mrr / n_commits
    fold_metrics.append(fold_result)



In [ ]:
[Lucene] Fold 1/10
[Lucene] Fold 2/10
[Lucene] Fold 3/10
[Lucene] Fold 4/10
[Lucene] Fold 5/10
[Lucene] Fold 6/10
[Lucene] Fold 7/10
[Lucene] Fold 8/10
[Lucene] Fold 9/10
[Lucene] Fold 10/10

In [ ]:
# Aggregate results across folds
cv_results = pd.DataFrame(fold_metrics)
summary = cv_results.agg(['mean','std'])

print("\n===== Lucene Baseline: 10-Fold CV Results =====")
print(summary)

In [ ]:

===== Lucene Baseline: 10-Fold CV Results =====
          HR@1      HR@3      HR@5     HR@10     HR@15       MRR
mean  0.383478  0.539136  0.604205  0.695764  0.747681  0.484421
std   0.048690  0.066492  0.060000  0.054201  0.046434  0.048603